In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

# LOAD DATA

train_df = pd.read_csv("dataset_B_training.csv")
test_df = pd.read_csv("dataset_B_testing.csv")

# ENCODE CATEGORICAL COLUMNS

combined = pd.concat([train_df.drop('h1n1_vaccine', axis=1), test_df])

for col in combined.columns:
    if combined[col].dtype == 'object':
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))

X_all = combined.iloc[:len(train_df)]
X_test = combined.iloc[len(train_df):]

y = train_df['h1n1_vaccine']

# AVERAGE (MEAN) IMPUTATION

imputer = SimpleImputer(strategy='mean')

X_all = pd.DataFrame(
    imputer.fit_transform(X_all),
    columns=X_all.columns
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

# TRAIN VALID SPLIT

X_train, X_valid, y_train, y_valid = train_test_split(
    X_all,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# LOGISTIC REGRESSION

model = LogisticRegression(max_iter=5000)

model.fit(X_train, y_train)

# ACCURACY & F1 SCORE

y_pred = model.predict(X_valid)

accuracy = accuracy_score(y_valid, y_pred)
f1 = f1_score(y_valid, y_pred)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", round(accuracy * 100, 2), "%")
print("F1 Score:", round(f1, 4))

# TRAIN FULL DATA

model.fit(X_all, y)

# TEST PREDICTIONS

test_predictions = model.predict(X_test)

# SUBMISSION FILE

submission = pd.DataFrame({
    'respondent_id': test_df['respondent_id'],
    'h1n1_vaccine': test_predictions
})

submission.to_csv('dataset_B_example_submission.csv', index=False)

print("submission.csv created successfully!")
print(submission.head())

Accuracy: 0.7457983193277311
Accuracy Percentage: 74.58 %
F1 Score: 0.6553
submission.csv created successfully!
   respondent_id  h1n1_vaccine
0           4757             0
1           4758             0
2           4759             0
3           4760             0
4           4761             0
